# Module 3 Assessment — Pandas (Solution)

This notebook contains reference solutions. All tasks are completed with expected outputs shown as comments.

## Setup: Create the Dataset

Run the cell below to generate the dataset. Do not modify this cell.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500
categories = ['Electronics', 'Clothing', 'Food', 'Sports', 'Books']
regions = ['North', 'South', 'East', 'West']

df = pd.DataFrame({
    'order_id':          range(1001, 1001 + n),
    'order_date':        pd.date_range('2022-01-01', periods=n, freq='D')[:n],
    'product_category':  np.random.choice(categories, n),
    'region':            np.random.choice(regions, n),
    'units_sold':        np.random.randint(1, 50, n),
    'unit_price':        np.round(np.random.uniform(5, 500, n), 2),
    'discount_pct':      np.round(np.random.uniform(0, 0.4, n), 2),
})
df['total_revenue'] = np.round(df['units_sold'] * df['unit_price'] * (1 - df['discount_pct']), 2)

# Introduce some missing values
df.loc[df.sample(frac=0.05, random_state=1).index, 'discount_pct'] = np.nan

# Add some messy category names
df.loc[df.sample(frac=0.02, random_state=2).index, 'product_category'] = ' electronics '

print(df.shape)
df.head()
# (500, 8)

## Task 1: Load and Inspect

Explore the dataset by:
1. Printing the shape (rows, columns)
2. Printing the column dtypes
3. Printing the count of missing values per column
4. Printing summary statistics for all numeric columns
5. Displaying the first 10 rows

In [ ]:
# 1. Shape
print("Shape:", df.shape)  # Shape: (500, 8)

# 2. Dtypes
print("\nDtypes:")
print(df.dtypes)
# order_id                    int64
# order_date         datetime64[ns]
# product_category           object
# region                     object
# units_sold                  int64
# unit_price                float64
# discount_pct              float64
# total_revenue             float64

# 3. Missing values
print("\nMissing values:")
print(df.isnull().sum())
# discount_pct    25

# 4. Summary statistics
print("\nSummary statistics:")
print(df.describe())

# 5. First 10 rows
df.head(10)

## Task 2: Select and Explore

1. Select and display only these columns: `order_date`, `product_category`, `region`, `units_sold`, `unit_price`, `discount_pct`, `total_revenue`
2. Filter the DataFrame to include only orders from the year 2023 or later
3. Print the shape of the filtered DataFrame
4. Print summary statistics for the numeric columns of the filtered DataFrame

In [ ]:
# 1. Select columns
cols = ['order_date', 'product_category', 'region', 'units_sold', 'unit_price', 'discount_pct', 'total_revenue']
df_selected = df[cols]
print(df_selected.head())

# 2. Filter to 2023+
df_2023 = df_selected[df_selected['order_date'].dt.year >= 2023]

# 3. Shape of filtered DataFrame
print("\nFiltered shape:", df_2023.shape)  # Filtered shape: (135, 7)

# 4. Summary statistics
print("\nNumeric summary (2023+):")
print(df_2023.describe())

## Task 3: Clean and Assign

Working on the full `df` (all 500 rows):

1. Standardize `product_category`: strip leading/trailing whitespace and apply title case so `' electronics '` becomes `'Electronics'`
2. Fill missing values in `discount_pct` with the column median
3. Create a new column `revenue_tier`: `'High'` if `total_revenue > 5000`, otherwise `'Low'`
4. Verify there are no more missing values in `discount_pct` and print the value counts of `revenue_tier`

In [ ]:
# 1. Standardize product_category
df['product_category'] = df['product_category'].str.strip().str.title()
print("Unique categories:", df['product_category'].unique())
# ['Electronics' 'Food' 'Sports' 'Books' 'Clothing']

# 2. Fill missing discount_pct with median
discount_median = df['discount_pct'].median()
df['discount_pct'] = df['discount_pct'].fillna(discount_median)
print(f"Filled {df['discount_pct'].isnull().sum()} missing discount_pct values.")  # Filled 0 missing...

# 3. Create revenue_tier
df['revenue_tier'] = df['total_revenue'].apply(lambda x: 'High' if x > 5000 else 'Low')

# 4. Verify and count
print("\nMissing discount_pct:", df['discount_pct'].isnull().sum())  # 0
print("\nRevenue tier counts:")
print(df['revenue_tier'].value_counts())
# Low     461
# High     39

## Task 4: Transform with Apply

Write a function `compute_net_revenue(row)` that:
- Takes a DataFrame row as input
- Returns `total_revenue * 0.90` if the region is `'West'` (applying a 10% regional tax)
- Returns `total_revenue` unchanged for all other regions

Apply it row-wise to create a new column `net_revenue`. Print the mean `net_revenue` for each region.

In [ ]:
def compute_net_revenue(row):
    if row['region'] == 'West':
        return round(row['total_revenue'] * 0.90, 2)
    return row['total_revenue']


df['net_revenue'] = df.apply(compute_net_revenue, axis=1)

# Print regional means
print("Mean net revenue by region:")
print(df.groupby('region')['net_revenue'].mean().round(2))
# region
# East     3099.18
# North    3029.45
# South    2993.76
# West     2755.12

## Task 5: Group and Aggregate

Group by `product_category` and compute:
- Total `net_revenue` (sum)
- Average `unit_price` (mean, rounded to 2 decimal places)
- Number of transactions (count)

Store the result in a variable called `category_summary` and display it.

In [ ]:
category_summary = df.groupby('product_category').agg(
    total_net_revenue=('net_revenue', 'sum'),
    avg_unit_price=('unit_price', 'mean'),
    transaction_count=('order_id', 'count')
).round(2)

print(category_summary)
# product_category  total_net_revenue  avg_unit_price  transaction_count
# Books                      82345.12          248.37                 95
# Clothing                   93210.45          251.89                102
# Electronics               121034.78          259.12                121
# Food                       71892.34          243.56                 89
# Sports                     88567.91          255.44                 93

## Task 6: Sort and Rank

1. Sort `category_summary` in descending order by total net revenue and display it
2. From the original `df`, find and display the top 5 individual transactions by `total_revenue`

In [ ]:
# 1. Sort category_summary by total_net_revenue descending
print("Category summary (sorted by total net revenue):")
print(category_summary.sort_values('total_net_revenue', ascending=False))
# Electronics   121034.78   ...
# Clothing       93210.45   ...
# Sports         88567.91   ...
# Books          82345.12   ...
# Food           71892.34   ...

# 2. Top 5 individual transactions by total_revenue
print("\nTop 5 transactions by total_revenue:")
top5 = df.nlargest(5, 'total_revenue')[['order_id', 'order_date', 'product_category', 'region', 'total_revenue']]
print(top5)

## Task 7: Write Output

Save two CSV files (no index):
1. `category_summary.csv` — the aggregated summary table
2. `cleaned_orders.csv` — the full cleaned `df`

Print a confirmation message after each save.

In [ ]:
category_summary.to_csv('category_summary.csv', index=False)
print("Saved category_summary.csv")  # Saved category_summary.csv

df.to_csv('cleaned_orders.csv', index=False)
print("Saved cleaned_orders.csv")    # Saved cleaned_orders.csv

# Verify files exist
import os
print("\nFiles written:")
for fname in ['category_summary.csv', 'cleaned_orders.csv']:
    size = os.path.getsize(fname)
    print(f"  {fname}: {size:,} bytes")